# Run Inference — T5 Spell Checker → INSERT predictions

**Pipeline**: Load model từ Drive → Query sentences từ DB qua ngrok → Predict trên GPU → INSERT predictions

## Yêu cầu
1. Model `t5_spell_checker/` đã có trong Drive (từ training)
2. Trên máy Windows: terminal chạy `ngrok tcp 5432`
3. Colab Pro với GPU T4

## Tốc độ
- CPU (local): ~30s/câu × 100 = 50 phút
- **GPU T4 (Colab): ~0.5s/câu × 100 = 1 phút** ⚡
- → Predict 5,000 câu chỉ mất ~10-15 phút trên Colab

## 1. Cài thư viện + Mount Drive

In [ ]:
!pip install -q psycopg2-binary transformers sentencepiece

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Config — sửa ngrok host:port + password

In [ ]:
import re, uuid, time
from datetime import datetime
from difflib import SequenceMatcher

import torch
import psycopg2
from psycopg2.extras import execute_values
from tqdm.auto import tqdm
from transformers import T5Tokenizer, T5ForConditionalGeneration

# ===== SỬA 2 GIÁ TRỊ NÀY THEO NGROK CỦA BẠN =====
DB_CONFIG = {
    'host':     '0.tcp.ap.ngrok.io',     # ← copy từ ngrok output
    'port':     16210,                    # ← copy từ ngrok output
    'dbname':   'postgres',
    'user':     'postgres',
    'password': '12345',
}
SCHEMA = 'public'

# Model path trong Drive
MODEL_PATH = '/content/drive/MyDrive/t5_spell_checker'
MODEL_NAME = 'T5-Base-Spell-Checker'   # phải khớp với model_name trong DB
MODEL_VERSION = '1.0.0'

# Inference settings
PREFIX = 'fix grammar and spelling: '
MAX_LEN = 128
BATCH_SIZE = 32                          # GPU có thể xử lý batch lớn
LIMIT = 5000                             # Số sentence tối đa (None = tất cả ~171k)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 3. Load model từ Drive

In [ ]:
print(f'Load model từ {MODEL_PATH}...')
tokenizer = T5Tokenizer.from_pretrained(MODEL_PATH)
model = T5ForConditionalGeneration.from_pretrained(MODEL_PATH)
model.to(DEVICE).eval()
print(f'✓ Model loaded ({sum(p.numel() for p in model.parameters()):,} parameters)')

## 4. Test connection DB qua ngrok

In [ ]:
# Test connection
try:
    conn = psycopg2.connect(**DB_CONFIG)
    with conn.cursor() as cur:
        cur.execute('SELECT version()')
        print(f'✓ DB connected: {cur.fetchone()[0][:50]}...')
        cur.execute(f'SELECT COUNT(*) FROM {SCHEMA}.sentences')
        print(f'  Total sentences in DB: {cur.fetchone()[0]:,}')
        cur.execute(f'SELECT COUNT(*) FROM {SCHEMA}.predictions')
        print(f'  Existing predictions:  {cur.fetchone()[0]:,}')
    conn.close()
except Exception as e:
    print(f'❌ DB connection failed: {e}')
    print('Kiểm tra: ngrok đang chạy? host/port/password đúng?')

## 5. Helper functions — phân loại lỗi tự động

In [ ]:
FUNCTION_WORDS = {
    'is','are','was','were','am','be','been','being',
    'has','have','had','having','do','does','did',
    'a','an','the','this','that','these','those',
    'in','on','at','to','for','of','with','by',
    'and','or','but','so','because','if','when',
}

def classify_error(src: str, tgt: str) -> int | None:
    """
    error_type_id:
      1 = SPELL, 2 = GRAMMAR, 3 = VOCAB, 4 = PUNCT, 5 = CAPITAL, 6 = WORD_ORDER
    """
    if src == tgt: return None
    s_words, t_words = src.split(), tgt.split()
    if sorted(s_words) == sorted(t_words): return 6   # word order
    if src.lower() == tgt.lower(): return 5            # capital
    s_clean = re.sub(r'[^\w\s]', '', src)
    t_clean = re.sub(r'[^\w\s]', '', tgt)
    if s_clean.strip() == t_clean.strip(): return 4    # punctuation

    # Tìm từ thay đổi
    matcher = SequenceMatcher(None, s_words, t_words)
    changed = []
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag in ('replace', 'delete', 'insert'):
            changed.append((' '.join(s_words[i1:i2]), ' '.join(t_words[j1:j2])))

    # Function word change → GRAMMAR
    for old, new in changed:
        if old.lower() in FUNCTION_WORDS or new.lower() in FUNCTION_WORDS:
            return 2

    # Typo (1-2 ký tự khác trong cùng từ) → SPELL
    for old, new in changed:
        if old and new and ' ' not in old and ' ' not in new:
            ratio = SequenceMatcher(None, old.lower(), new.lower()).ratio()
            if ratio > 0.6:
                return 1
    return 3   # vocab default


def calc_confidence(src: str, tgt: str) -> float:
    if not src or not tgt: return 0.5
    ratio = SequenceMatcher(None, src.lower(), tgt.lower()).ratio()
    return round(0.7 + 0.3 * ratio, 4)


def predict_batch(sentences):
    out = []
    for i in range(0, len(sentences), BATCH_SIZE):
        batch = [PREFIX + str(s) for s in sentences[i:i+BATCH_SIZE]]
        enc = tokenizer(batch, return_tensors='pt', padding=True,
                        truncation=True, max_length=MAX_LEN).to(DEVICE)
        with torch.no_grad():
            ids = model.generate(**enc, max_length=MAX_LEN, num_beams=4)
        out.extend(tokenizer.batch_decode(ids, skip_special_tokens=True))
    return out

print('✓ Helper functions ready')

## 6. Query sentences chưa có prediction

In [ ]:
conn = psycopg2.connect(**DB_CONFIG)

# Tìm model_id trong DB (đã register từ training trước)
with conn.cursor() as cur:
    cur.execute(
        f"SELECT model_id FROM {SCHEMA}.models WHERE model_name=%s AND version=%s ORDER BY created_at DESC LIMIT 1",
        (MODEL_NAME, MODEL_VERSION)
    )
    row = cur.fetchone()
    if not row:
        raise ValueError(f'Không tìm thấy model {MODEL_NAME} v{MODEL_VERSION} trong DB. Hãy chạy register trước.')
    model_id = row[0]
    print(f'Model_id: {model_id}')

# Query sentences chưa có prediction từ model này
with conn.cursor() as cur:
    sql = f"""
        SELECT s.sentence_id, s.content
        FROM {SCHEMA}.sentences s
        WHERE NOT EXISTS (
            SELECT 1 FROM {SCHEMA}.predictions p
            WHERE p.sentence_id = s.sentence_id AND p.model_id = %s
        )
        ORDER BY s.created_at
    """
    if LIMIT:
        sql += f' LIMIT {LIMIT}'
    cur.execute(sql, (model_id,))
    rows = cur.fetchall()

print(f'Số sentence cần predict: {len(rows):,}')
if not rows:
    print('Tất cả sentences đã có prediction. Không có gì để làm.')

## 7. Inference + INSERT predictions

In [ ]:
if rows:
    sentence_ids = [r[0] for r in rows]
    contents = [r[1] for r in rows]

    # Predict theo batch lớn
    BATCH_OUTER = BATCH_SIZE * 4
    predictions = []

    for i in tqdm(range(0, len(contents), BATCH_OUTER), desc='Predict (GPU)'):
        batch_src = contents[i:i+BATCH_OUTER]
        batch_tgt = predict_batch(batch_src)
        for sid, src, tgt in zip(sentence_ids[i:i+BATCH_OUTER], batch_src, batch_tgt):
            src_clean, tgt_clean = src.strip(), tgt.strip()
            has_error = (src_clean.lower() != tgt_clean.lower())
            err_type = classify_error(src_clean, tgt_clean) if has_error else None
            conf = calc_confidence(src_clean, tgt_clean)
            predictions.append((
                str(uuid.uuid4()), sid, model_id,
                1 if has_error else 0, conf, datetime.now(),
                err_type, tgt_clean if has_error else None,
            ))

    # Batch INSERT
    print(f'\nINSERT {len(predictions):,} predictions vào DB...')
    with conn:
        with conn.cursor() as cur:
            execute_values(
                cur,
                f"""INSERT INTO {SCHEMA}.predictions
                    (prediction_id, sentence_id, model_id, label, confidence,
                     predicted_at, error_type_id, corrected_text)
                    VALUES %s""",
                predictions
            )
    print('✓ OK!')

    # Stats
    n_err = sum(1 for p in predictions if p[3] == 1)
    print(f'\n=== KẾT QUẢ ===')
    print(f'  Câu ĐÚNG (label=0):  {len(predictions) - n_err:,}')
    print(f'  Câu LỖI  (label=1):  {n_err:,}')
    print(f'  Tổng:                 {len(predictions):,}')

conn.close()

## 8. Verify trong DB

In [ ]:
conn = psycopg2.connect(**DB_CONFIG)
with conn.cursor() as cur:
    cur.execute(f"""
        SELECT et.name, COUNT(*) FROM {SCHEMA}.predictions p
        LEFT JOIN {SCHEMA}.error_types et ON p.error_type_id = et.error_type_id
        WHERE p.model_id = %s AND p.label = 1
        GROUP BY et.name ORDER BY COUNT(*) DESC
    """, (model_id,))
    print('Phân bố lỗi theo loại:')
    for name, n in cur.fetchall():
        print(f'  {name or "(unknown)":<25} {n:,}')
conn.close()

---
## Tốc độ thực tế trên GPU T4

| Limit | CPU local | GPU Colab T4 |
|---|---|---|
| 100 câu | 5-10 phút | **30-60 giây** |
| 1,000 câu | 50-100 phút | **3-5 phút** |
| 5,000 câu | 4-8 giờ | **15-25 phút** |
| 171,000 câu | nhiều ngày | **6-10 giờ** |

→ GPU nhanh hơn CPU **~20-50x** cho task này.

## Lưu ý ngrok

- **Đừng đóng terminal ngrok** trong lúc inference — đóng = mất connection
- Free ngrok URL **đổi mỗi lần restart** → phải update host:port nếu restart
- Free plan giới hạn 1 connection cùng lúc